In [5]:
import os
import re
import pandas as pd


def get_column_case_insensitive(df: pd.DataFrame, possible_names: list) -> str:
    """Finds the actual column name in DataFrame matching any string in possible_names."""
    col_map = {str(col).strip().lower(): col for col in df.columns}
    for name in possible_names:
        if name.lower().strip() in col_map:
            return col_map[name.lower().strip()]
    return None


def process_mdb_data():
    # 1. Read file path from environment variable MDB
    file_path = os.getenv("MDB")
    if not file_path:
        raise ValueError(
            "⚠️ MDB environment variable is not set. Please set it before running the script."
        )

    if not file_path.lower().endswith((".xlsx", ".xls")):
        file_path += ".xlsx"

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"⚠️ File not found at path: {file_path}")

    print(f"📖 Reading file from: {file_path}")
    df = pd.read_excel(file_path)

    # Detect exact column names dynamically
    sector_col = get_column_case_insensitive(df, ["Sector", "Sector Name"])
    location_col = get_column_case_insensitive(
        df, ["Location", "Project Location", "Site Location"]
    )
    activity_col = get_column_case_insensitive(
        df, ["Activity Description", "Activity", "Activity_Description"]
    )

    # Column Validation
    missing_cols = []
    if not sector_col:
        missing_cols.append("Sector")
    if not location_col:
        missing_cols.append("Location")
    if not activity_col:
        missing_cols.append("Activity Description")

    if missing_cols:
        raise KeyError(
            f"❌ Required column(s) missing from Excel: {missing_cols}. Available columns are: {list(df.columns)}"
        )

    # 2. Define list of Sectors to filter OUT
    sectors_to_exclude = [
        "INFRA-2",
        "Coal Mining",
        "INFRA-1",
        "River Valley and Hydroelectric Projects",
    ]

    # 3. Target Activity Descriptions
    target_activities = [
        "1(a) Mining of minerals",
        "1(b) Off-shore and onshore oil and gas exploration, development and production",
        "1(d) Thermal Power Plants",
        "2(b) Mineral beneficiation",
        "2(c) Pellet Plant",
        "3(a) Metallurgical Industries (ferrous and non ferrous)",
        "3(b) Cement plants",
        "4(a) Petroleum refining industry",
        "4(b) Coke oven plants",
        "4(b)(ii) Coaltar processing units",
        "4(c) Asbestos milling / asbestos-based products",
        "4(d) Chlor-alkali industry",
        "4(e) Soda ash Industry",
        "4(f) Skin/hide processing including the tanning industry",
        "5(a) Chemical fertilizers",
        "5(b) Pesticides industry and pesticide specific intermediates (excluding formulations)",
        "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions",
        "5(d) Manmade fibers manufacturing",
        "5(e ) Petroleum products and petrochemical based processing such as production of carbon black and electrode grade graphite (processes other than cracking",
        "5(f) Synthetic organic chemicals industry",
        "5(g) Distilleries",
        "5(ga) Grain based distilleries",
        "5(h) Integrated paint industry",
        "5(i) Pulp & Paper Industry",
        "5(j) Sugar Industry",
        "6(a) Pipelines",
    ]

    # Activity ID Lookup Dictionary (ID -> Description)
    activity_mapping = {
        "15": "4(c) Asbestos milling / asbestos-based products",
        "11": "3(b) Cement plants",
        "19": "5(a) Chemical fertilizers",
        "16": "4(d) Chlor-alkali industry",
        "14": "4(b)(ii) Coaltar processing units",
        "13": "4(b) Coke oven plants",
        "25": "5(g) Distilleries",
        "75": "5(ga) Grain based distilleries",
        "26": "5(h) Integrated paint industry",
        "22": "5(d) Manmade fibers manufacturing",
        "10": "3(a) Metallurgical Industries (ferrous and non ferrous)",
        "1": "1(a) Mining of minerals",
        "9": "2(b) Mineral beneficiation",
        "7": "1(e) Nuclear power projects and processing of nuclear fuel",
        "3": "1(b) Off-shore and onshore oil and gas exploration, development and production",
        "79": "2(c) Pellet Plant",
        "20": "5(b) Pesticides industry and pesticide specific intermediates (excluding formulations)",
        "21": "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions",
        "23": "5(e ) Petroleum products and petrochemical based processing such as production of carbon black and electrode grade graphite (processes other than cracking",
        "12": "4(a) Petroleum refining industry",
        "2": "6(a) Pipelines",
        "27": "5(i) Pulp & Paper Industry",
        "30": "7(b) Ship breaking yards including ship breaking units",
        "18": "4(f) Skin/hide processing including the tanning industry",
        "77": "1(a)(ii) Slurry pipelines passing through national parks / sanctuaries / coral reefs, ecologically sensitive areas",
        "17": "4(e) Soda ash Industry",
        "28": "5(j) Sugar Industry",
        "24": "5(f) Synthetic organic chemicals industry",
        "6": "1(d) Thermal Power Plants",
    }

    # Build reverse lookup map (Normalized Description -> ID)
    desc_to_id = {
        re.sub(r"\s+", " ", desc).strip().lower(): act_id
        for act_id, desc in activity_mapping.items()
    }

    # Step 2: Filter OUT specified Sectors
    filtered_df = df[
        ~df[sector_col].astype(str).str.strip().isin(sectors_to_exclude)
    ].copy()

    # Step 3: Match Activity Descriptions
    target_activities_normalized = set(
        re.sub(r"\s+", " ", act).strip().lower() for act in target_activities
    )

    filtered_df["_activity_norm"] = (
        filtered_df[activity_col]
        .astype(str)
        .apply(lambda x: re.sub(r"\s+", " ", x).strip().lower())
    )

    matching_df = filtered_df[
        filtered_df["_activity_norm"].isin(target_activities_normalized)
    ].copy()

    # Get distinct combinations of Location and Activity Description
    distinct_combos = matching_df[
        [location_col, activity_col, "_activity_norm"]
    ].drop_duplicates()

    # Step 4: Map to the Activity ID and format with a trailing comma
    distinct_combos["Activity ID"] = distinct_combos["_activity_norm"].map(desc_to_id)

    # Format into key-value style mapping output: "15": "4(c) Asbestos milling / asbestos-based products",
    distinct_combos["Activity Mapping"] = distinct_combos.apply(
        lambda row: f'"{row["Activity ID"]}": "{row[activity_col]}",'
        if pd.notna(row["Activity ID"])
        else "",
        axis=1,
    )

    # Drop temporary normalization helper column
    distinct_combos = distinct_combos.drop(columns=["_activity_norm"])

    # Step 5: Save all locations combined into a SINGLE Excel output file
    output_directory = os.path.dirname(file_path)
    output_file = os.path.join(
        output_directory, "Mapped_Activities_All_Locations.xlsx"
    )

    distinct_combos.to_excel(output_file, index=False)
    print(
        f"✅ Successfully exported all {len(distinct_combos)} distinct location records to single file: {output_file}"
    )


if __name__ == "__main__":
    process_mdb_data()

📖 Reading file from: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\MasterDB.xlsx
✅ Successfully exported all 343 distinct location records to single file: F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\Mapped_Activities_All_Locations.xlsx
